# 🧠 Document Autoencoder — Training Notebook

**U-Net style convolutional autoencoder** for sharp receipt image reconstruction.

| Setting | Value |
|---|---|
| Dataset | ICDAR-2019 SROIE receipts |
| Image size | 512 × 512 |
| Loss | SSIM+L1 + VGG Perceptual + Sobel Edge |
| Optimizer | AdamW + CosineAnnealingWarmRestarts |
| Early stopping | Val SSIM (patience = 15) |


## 1 · Environment Setup

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU     : {props.name}')
    print(f'VRAM    : {props.total_memory / 1024**3:.1f} GB')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')

## 2 · Configuration

In [ ]:
from autoencoder.config import (
    DEVICE, DATA_DIR, CHECKPOINT_DIR, OUTPUT_DIR,
    IMG_SIZE, BATCH_SIZE, NUM_WORKERS,
    LEARNING_RATE, WEIGHT_DECAY, EPOCHS,
    GRAD_CLIP_MAX_NORM, SCHEDULER_T0, SCHEDULER_T_MULT,
    EARLY_STOP_PATIENCE, SAVE_GRID_EVERY,
    LAMBDA_SSIM_L1, LAMBDA_PERCEPTUAL, LAMBDA_EDGE,
    SSIM_WEIGHT, L1_WEIGHT,
)
print(f'Data dir      : {DATA_DIR}')
print(f'Checkpoints   : {CHECKPOINT_DIR}')
print(f'Image size    : {IMG_SIZE}x{IMG_SIZE}')
print(f'Batch size    : {BATCH_SIZE}')
print(f'Epochs        : {EPOCHS}')
print(f'LR            : {LEARNING_RATE}')
print(f'Early stop    : patience={EARLY_STOP_PATIENCE}')
print(f'Loss weights  : SSIM+L1={LAMBDA_SSIM_L1}  Perceptual={LAMBDA_PERCEPTUAL}  Edge={LAMBDA_EDGE}')

## 3 · Dataset Inspection

In [ ]:
from autoencoder.dataset import get_dataloaders
train_loader, val_loader = get_dataloaders()
print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
inputs, targets = next(iter(train_loader))
print(f'Batch shape   : {inputs.shape}')
print(f'Value range   : [{inputs.min():.3f}, {inputs.max():.3f}]')

In [ ]:
import matplotlib.pyplot as plt
import torchvision.utils as vutils
grid = vutils.make_grid(inputs[:4], nrow=4, padding=4, pad_value=1.0)
fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.axis('off')
ax.set_title('Sample training images (augmented inputs)', fontsize=13)
plt.tight_layout(); plt.show()

## 4 · Model Architecture

In [ ]:
from autoencoder.model import DocumentAutoencoder, count_parameters
model = DocumentAutoencoder().to(DEVICE)
n_params = count_parameters(model)
print(f'Trainable parameters: {n_params:,}  ({n_params / 1e6:.2f} M)')
dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f'Input  : {dummy.shape}')
print(f'Output : {out.shape}')
assert dummy.shape == out.shape
print('Shape check passed')

In [ ]:
bn, skips = model.encode(dummy)
print('Encoder skip connection shapes:')
for i, s in enumerate(skips):
    print(f'  skip[{i}] : {tuple(s.shape)}')
print(f'  bottleneck : {tuple(bn.shape)}')

## 5 · Loss Functions

In [ ]:
from autoencoder.losses import CompositeLoss
loss_fn = CompositeLoss().to(DEVICE)
pred_t   = torch.rand(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
target_t = torch.rand(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
losses = loss_fn(pred_t, target_t)
print('Loss components (sanity check on random tensors):')
for k, v in losses.items():
    val = v.item() if torch.is_tensor(v) else v
    print(f'  {k:<15} : {val:.6f}')

## 6 · LR Schedule Preview

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
lrs = []
_opt = AdamW([torch.zeros(1, requires_grad=True)], lr=LEARNING_RATE)
_sch = CosineAnnealingWarmRestarts(_opt, T_0=SCHEDULER_T0, T_mult=SCHEDULER_T_MULT)
for _ in range(EPOCHS):
    lrs.append(_opt.param_groups[0]['lr'])
    _sch.step()
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(range(1, EPOCHS+1), lrs, lw=2, color='#4c87e0')
ax.set_xlabel('Epoch'); ax.set_ylabel('LR')
ax.set_title('CosineAnnealingWarmRestarts LR Schedule')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 7 · Training Utilities

In [ ]:
import time
import torch.nn as nn
from tqdm.notebook import tqdm
from torchvision.transforms.functional import to_pil_image

# AMP scaler for mixed-precision training (float16 on GPU, ~1.5-2x faster)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')

def compute_psnr(pred, target):
    mse = torch.mean((pred - target) ** 2).item()
    if mse < 1e-10: return 100.0
    return 10 * torch.log10(torch.tensor(1.0 / mse)).item()

def save_comparison_grid(model, val_loader, epoch, device, num_samples=6):
    model.eval()
    originals, reconstructed = [], []
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            originals.append(targets.cpu())
            reconstructed.append(outputs.cpu())
            if sum(o.size(0) for o in originals) >= num_samples: break
    originals     = torch.cat(originals,     0)[:num_samples]
    reconstructed = torch.cat(reconstructed, 0)[:num_samples]
    comparison = torch.stack([originals, reconstructed], 1).view(-1, *originals.shape[1:])
    grid = vutils.make_grid(comparison, nrow=2, padding=4, pad_value=0.5)
    path = os.path.join(OUTPUT_DIR, f'reconstruction_epoch_{epoch:03d}.png')
    to_pil_image(grid).save(path)
    print(f'  [Grid] Saved -> {path}')
    return path

def train_one_epoch(model, loader, loss_fn, optimizer, device, epoch, scaler):
    model.train()
    totals = {'total': 0.0, 'ssim_l1': 0.0, 'perceptual': 0.0, 'edge': 0.0}
    n = 0
    pbar = tqdm(loader, desc=f'Epoch {epoch:3d} [Train]', leave=False)
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        # AMP autocast: runs forward pass in float16 where safe
        with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
            losses = loss_fn(model(inputs), targets)
        # Scaled backward + optimizer step
        scaler.scale(losses['total']).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_MAX_NORM)
        scaler.step(optimizer)
        scaler.update()
        for k in totals: totals[k] += losses[k].item() if torch.is_tensor(losses[k]) else losses[k]
        n += 1
        pbar.set_postfix(loss=f"{losses['total'].item():.4f}", ssim=f"{losses['ssim_raw']:.4f}")
    return {k: v/n for k, v in totals.items()}

@torch.no_grad()
def validate(model, loader, loss_fn, device):
    model.eval()
    totals = {'total': 0.0, 'ssim_l1': 0.0, 'perceptual': 0.0, 'edge': 0.0, 'ssim_raw': 0.0}
    total_psnr = 0.0; n = 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
            outputs = model(inputs)
            losses = loss_fn(outputs, targets)
        for k in totals: totals[k] += losses[k].item() if torch.is_tensor(losses[k]) else losses[k]
        total_psnr += compute_psnr(outputs.float(), targets.float()); n += 1
    avg = {k: v/n for k, v in totals.items()}
    avg['psnr'] = total_psnr / n
    return avg

print('Training utilities ready (AMP enabled for GPU)')

## 8 · Training Loop
> Run this cell to start training.

In [ ]:
model     = DocumentAutoencoder().to(DEVICE)
loss_fn   = CompositeLoss().to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=SCHEDULER_T0, T_mult=SCHEDULER_T_MULT)
scaler    = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')  # AMP scaler

best_ssim = 0.0; patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_ssim': [], 'val_psnr': []}

print('=' * 70)
print('  Starting training (AMP + 256px + batch=8)...')
print('=' * 70)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_losses = train_one_epoch(model, train_loader, loss_fn, optimizer, DEVICE, epoch, scaler)
    scheduler.step()
    val_losses = validate(model, val_loader, loss_fn, DEVICE)
    val_ssim = val_losses['ssim_raw']; val_psnr = val_losses['psnr']
    elapsed = time.time() - t0

    history['train_loss'].append(train_losses['total'])
    history['val_loss'].append(val_losses['total'])
    history['val_ssim'].append(val_ssim)
    history['val_psnr'].append(val_psnr)

    print(f"  Epoch {epoch:3d}/{EPOCHS} | Train: {train_losses['total']:.4f} | Val: {val_losses['total']:.4f} | SSIM: {val_ssim:.4f} | PSNR: {val_psnr:.2f}dB | LR: {optimizer.param_groups[0]['lr']:.2e} | {elapsed:.1f}s")
    print(f"           | SSIM+L1: {val_losses['ssim_l1']:.4f} | Perceptual: {val_losses['perceptual']:.4f} | Edge: {val_losses['edge']:.4f}")

    if epoch % SAVE_GRID_EVERY == 0 or epoch == 1:
        save_comparison_grid(model, val_loader, epoch, DEVICE)

    if val_ssim > best_ssim:
        best_ssim = val_ssim; patience_counter = 0
        ckpt_path = os.path.join(CHECKPOINT_DIR, 'best_autoencoder.pth')
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'scaler_state_dict': scaler.state_dict(),
                    'best_ssim': best_ssim, 'val_psnr': val_psnr, 'history': history}, ckpt_path)
        print(f'  * New best SSIM {best_ssim:.4f} - checkpoint saved!')
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f'  Early stopping after {EARLY_STOP_PATIENCE} epochs without improvement.')
            break
    print()

final_path = os.path.join(CHECKPOINT_DIR, 'final_autoencoder.pth')
torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'best_ssim': best_ssim, 'history': history}, final_path)

print('=' * 70)
print(f'  Training complete!  Best SSIM: {best_ssim:.4f}')
print(f'  Best checkpoint  -> {os.path.join(CHECKPOINT_DIR, "best_autoencoder.pth")}')
print(f'  Final checkpoint -> {final_path}')
print('=' * 70)

## 9 · Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(epochs_ran, history['train_loss'], label='Train', lw=2, color='#4c87e0')
axes[0].plot(epochs_ran, history['val_loss'],   label='Val',   lw=2, color='#e05c4c', linestyle='--')
axes[0].set_title('Total Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs_ran, history['val_ssim'], lw=2, color='#4cad62')
axes[1].set_title('Val SSIM (higher is better)'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1); axes[1].grid(True, alpha=0.3)
axes[2].plot(epochs_ran, history['val_psnr'], lw=2, color='#9b59b6')
axes[2].set_title('Val PSNR dB (higher is better)'); axes[2].set_xlabel('Epoch')
axes[2].grid(True, alpha=0.3)
plt.tight_layout()
curves_path = os.path.join(OUTPUT_DIR, 'training_curves.png')
fig.savefig(curves_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved -> {curves_path}')

## 10 · Visualise Reconstructions

In [ ]:
best_ckpt = os.path.join(CHECKPOINT_DIR, 'best_autoencoder.pth')
ckpt = torch.load(best_ckpt, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, SSIM {ckpt['best_ssim']:.4f})")

In [ ]:
NUM_VIS = 4
all_inputs, all_targets, all_outputs = [], [], []
with torch.no_grad():
    for inp, tgt in val_loader:
        inp, tgt = inp.to(DEVICE), tgt.to(DEVICE)
        out = model(inp)
        all_inputs.append(inp.cpu()); all_targets.append(tgt.cpu()); all_outputs.append(out.cpu())
        if sum(x.size(0) for x in all_inputs) >= NUM_VIS: break
inp_t = torch.cat(all_inputs,  0)[:NUM_VIS]
tgt_t = torch.cat(all_targets, 0)[:NUM_VIS]
out_t = torch.cat(all_outputs, 0)[:NUM_VIS]

rows = []
for i in range(NUM_VIS): rows.extend([inp_t[i], tgt_t[i], out_t[i]])
grid = vutils.make_grid(torch.stack(rows), nrow=3, padding=6, pad_value=0.5)
fig, ax = plt.subplots(figsize=(12, 5 * NUM_VIS // 2))
ax.imshow(grid.permute(1, 2, 0))
ax.axis('off')
ax.set_title('Input (augmented)  |  Target (clean)  |  Reconstructed', fontsize=13)
plt.tight_layout(); plt.show()

from autoencoder.losses import SSIMLoss
ssim_fn = SSIMLoss()
print('Per-image SSIM (reconstructed vs target):')
for i in range(NUM_VIS):
    s = ssim_fn.compute_ssim_value(out_t[i:i+1], tgt_t[i:i+1])
    print(f'  Image {i+1} : {s:.4f}')

## 11 · Resume Training (Optional)
> Run **instead of Section 8** to continue from a saved checkpoint.

In [ ]:
RESUME_FROM = os.path.join(CHECKPOINT_DIR, 'best_autoencoder.pth')
if not os.path.exists(RESUME_FROM):
    raise FileNotFoundError(f'Checkpoint not found: {RESUME_FROM}')

ckpt = torch.load(RESUME_FROM, map_location=DEVICE)
model     = DocumentAutoencoder().to(DEVICE)
loss_fn   = CompositeLoss().to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=SCHEDULER_T0, T_mult=SCHEDULER_T_MULT)

model.load_state_dict(ckpt['model_state_dict'])
optimizer.load_state_dict(ckpt['optimizer_state_dict'])
if 'scheduler_state_dict' in ckpt: scheduler.load_state_dict(ckpt['scheduler_state_dict'])

start_epoch = ckpt['epoch'] + 1
best_ssim   = ckpt['best_ssim']
history     = ckpt.get('history', {'train_loss': [], 'val_loss': [], 'val_ssim': [], 'val_psnr': []})
patience_counter = 0

print(f"Resumed from epoch {ckpt['epoch']}  (best SSIM so far: {best_ssim:.4f})")
print(f'Continuing from epoch {start_epoch} to {EPOCHS}')

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()
    train_losses = train_one_epoch(model, train_loader, loss_fn, optimizer, DEVICE, epoch)
    scheduler.step()
    val_losses = validate(model, val_loader, loss_fn, DEVICE)
    val_ssim = val_losses['ssim_raw']; val_psnr = val_losses['psnr']
    elapsed = time.time() - t0
    history['train_loss'].append(train_losses['total'])
    history['val_loss'].append(val_losses['total'])
    history['val_ssim'].append(val_ssim)
    history['val_psnr'].append(val_psnr)
    print(f"  Epoch {epoch:3d}/{EPOCHS} | Train: {train_losses['total']:.4f} | Val: {val_losses['total']:.4f} | SSIM: {val_ssim:.4f} | PSNR: {val_psnr:.2f}dB | {elapsed:.1f}s")
    if epoch % SAVE_GRID_EVERY == 0:
        save_comparison_grid(model, val_loader, epoch, DEVICE)
    if val_ssim > best_ssim:
        best_ssim = val_ssim; patience_counter = 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'best_ssim': best_ssim, 'val_psnr': val_psnr, 'history': history},
                   os.path.join(CHECKPOINT_DIR, 'best_autoencoder.pth'))
        print(f'  * New best SSIM {best_ssim:.4f} - checkpoint saved!')
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print('  Early stopping triggered.'); break
    print()

print(f'Done!  Best SSIM: {best_ssim:.4f}')